# GameTheory-16b : Automated Mechanism Design (AMD)

**Navigation** : [Version Python](GameTheory-16b-Automated-Mechanism-Design-Python.ipynb) | [<< 16-MechanismDesign](GameTheory-16-MechanismDesign-Python.ipynb) | [Index GameTheory](README.md) | [17-MultiAgent-RL >>](GameTheory-17-MultiAgent-RL-Python.ipynb) | [jumeau Rust](GameTheory-16b-Automated-Mechanism-Design-Rust.ipynb)

## Le saut de type

Le design de mécanismes classique cherche un joli mécanisme **général**, conçu à la main pour une classe de situations. L'**Automated Mechanism Design** (Conitzer & Sandholm) fait autre chose : on spécifie les types possibles, les issues, les utilités, l'objectif du designer et les contraintes d'incitation — et **on calcule le mécanisme adapté à l'instance**.

```
jeu ordinaire     :  G fixé,   a_i ∈ A_i
un cran au-dessus :  G fixé,   π_i ∈ Π_i
AMD               :  G ∈ 𝒢   devient une VARIABLE DE DÉCISION
                     M* = argmax_{M ∈ ℳ} J(M)   s.c.  IC(M), IR(M), budget(M), …
```

> **Ce qui était auparavant l'ENVIRONNEMENT des agents devient l'OBJET manipulé au niveau suivant.**

## La question renversée

On ne demande plus « **quelle action dois-je choisir ?** » mais
« **CONSTRUIS-MOI UN MONDE DE RÈGLES DANS LEQUEL LA PROPRIÉTÉ DÉSIRÉE DEVIENT VRAIE — PUIS PROUVE-MOI QUE TU L'AS RÉELLEMENT CONSTRUIT.** »

## Domaine fini, strictement

Deux agents, deux types chacun, deux issues. C'est ce qui rend le problème tractable : l'énumération suffit à engendrer le mécanisme-témoin.

```
(Θ, O, U, J, contraintes)  ⟶  M  ⟶  certificat
```

## Trois exemples guidés

Ces trois blocs sont des **solutions complètes** : ils montrent le geste. Les trois **exercices** qui les suivent (un par bloc, chacun placé juste après son exemple) reprennent le même outillage sur un paramètre que l'exemple a laissé fixe.

1. **Le générateur** — énumérer les mécanismes du domaine fini et sélectionner celui qui maximise `J` sous `DSIC` et `IR`. Sortir `M` explicitement (table de décision et paiements).
2. **La vérification** — vérifier `DSIC(M)`, `IR(M)`, `J(M) = J*` **indépendamment du générateur** : le vérificateur ne doit pas réutiliser le code qui a produit `M`.
3. **L'impossibilité** — durcir les contraintes jusqu'à ce qu'aucun mécanisme n'existe, et sortir alors soit un **témoin d'impossibilité**, soit une **déviation profitable** pour chaque candidat. Un échec sans témoin n'est pas un résultat.

## La frontière honnête

AMD optimise **dans un espace de mécanismes DONNÉ**. La question de strate 7 est : **comment apparaît une coordonnée qui n'appartenait pas encore à cet espace ?** Sandholm nous emmène très loin dans la strate 7, **mais il ne nous dispense pas de la strate 7.** Le notebook l'écrit en clôture.

***

## Prérequis

- `GameTheory-16-MechanismDesign-Python.ipynb` — théorie classique, Vickrey/VCG
- Notions de types bayésiens (`GameTheory-11-BayesianGames-Python`)
- C# : `record`, types locaux, LINQ, `List`/`Dictionary`

## Durée estimée : 35 minutes

***



In [1]:
// Cellule 1 — Imports et types de base
// Une Action est un couple (issue choisie, type reporté) ; un Mécanisme est
// une table : profil reporté -> (issue, paiement).
//
// Domaine : 2 agents, types θ ∈ {0, 1}, issue o ∈ {0, 1}, paiements ∈ {0, 1, 2}.
// Utilité : u_i(o, θ_i) = θ_i * o (l'agent de type 1 veut o=1, type 0 est indifférent).
//
// Bornes pédagogiques :
// - 4 profils (2^2)
// - 2^4 = 16 tables d'issue
// - 3^2 = 9 paiements par profil × 4 profils = 6561 tables de paiement
// - Total = 16 × 6561 = 104 976 candidats à énumérer
//
// Ce que cette version C# ajoute par rapport à la source : un VOCABULAIRE de
// données (types, constantes, ordre des profils) strictement séparé de toute
// logique. Le générateur (cellule 3) et le vérificateur (cellule 5) déclarent
// CHACUN leur type de mécanisme et CHACUN leurs primitives : la séparation
// exigée par la source tient ici par construction, et non par convention.

using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// Profil de types : l'ordre est celui du produit cartésien Python
// (itertools.product) : (0,0), (0,1), (1,0), (1,1).
// L'index d'un profil vaut 2 * type_0 + type_1.
static class Domaine
{
    public const int N_AGENTS = 2;
    public const int N_TYPES = 2;
    public const int N_ISSUES = 2;

    // Paiements du domaine complet ; la version compacte du générateur
    // (cellule 3) se restreint à {0, 1}.
    public static readonly int[] PaymentRange = { 0, 1, 2 };

    // Les 4 profils de types, dans l'ordre du produit cartésien.
    public static readonly List<int[]> Profiles = new List<int[]>
    {
        new[] { 0, 0 }, new[] { 0, 1 }, new[] { 1, 0 }, new[] { 1, 1 },
    };

    // Index d'un profil à partir des types reportés.
    public static int Index(int[] types) => 2 * types[0] + types[1];

    // Les types reportés d'un profil, dans l'ordre des agents.
    public static int[] Types(int index) => new[] { (index >> 1) & 1, index & 1 };

    // Utilité de l'agent i quand l'issue est tirée et que son type VRAI est theta_i.
    public static int UtilityI(int i, int outcome, int thetaI) => thetaI * outcome;
}


## Exemple guidé 1 — Le générateur

**Énoncé** : sur un domaine à 2 agents, 2 types chacun, 2 issues, avec un objectif `J` de bien-être social (`sum u_i(outcome, θ_i)`), implémentez `generate_amd` qui énumère les mécanismes et retourne celui qui maximise `J` sous DSIC + IR + budget non-négatif.

Indices :
- Énumérer **tous** les couples (table_issue, table_paiements) où les paiements sont dans `{0, 1, 2}`.
- Pour chaque mécanisme candidat, vérifier DSIC et IR (voir cellules suivantes).
- Sélectionner l'optimal sur l'objectif `J`.
- Retourner la **table explicite** (deux listes d'indexation par profil de types).

Domaine :
- 2 agents, types θ ∈ {0, 1} pour chaque agent.
- Issue : `o ∈ {0, 1}` (un bien public binaire).
- Utilité : `u_i(o, θ_i) = θ_i * o`.

**Sortie attendue** : `M*` affiché en toutes lettres (issue_table, payment_table), puis `J*` recalculé indépendamment.



In [2]:
// Cellule 3 — Implémentation du générateur (version compacte pour H.3 < 30s)
//
// Domaine : 2 agents, 2 types, 2 issues, paiements ∈ {0, 1}.
// (Réduit depuis {0, 1, 2} pour rester tractable : 16 issues × 256 paiements = 4096 candidats.)
// On garde l'esprit "énumération + tri" mais l'espace est plus petit.
//
// Le type du mécanisme est DÉCLARÉ ICI, avec le générateur : le vérificateur de
// la cellule 5 ne le connaît pas et travaille avec le sien.

record MecanismeGen(int[] Issues, int[,] Paiements);

static class Generateur
{
    // Copie locale de l'utilité : le générateur ne lit aucune primitive du
    // vérificateur, et inversement.
    static int UtilityI(int i, int outcome, int thetaI) => thetaI * outcome;

    // Les 16 tables d'issue, dans l'ordre du produit cartésien Python : le
    // profil d'index k occupe les bits (3-k) et (2-k) du masque.
    public static int[] Issues(int masque) => Enumerable.Range(0, 4).Select(r => (masque >> (3 - r)) & 1).ToArray();

    // Énumère tous les mécanismes du domaine (2 agents, 2 types, 2 issues,
    // paiements ∈ {0,1}) et sélectionne l'argmax sur le bien-être social.
    public static (MecanismeGen mecanisme, int bienEtre) Generate(int[] trueTypes)
    {
        int bestJ = -1;
        int[] bestIssues = Issues(0);
        int profileVrai = Domaine.Index(trueTypes);

        for (int masque = 0; masque < 16; masque++)
        {
            int[] issues = Issues(masque);
            // J = sum_i θ_i * issue[profil vrai] : l'issue lue au profil sincère.
            int j = trueTypes.Sum(theta => UtilityI(0, issues[profileVrai], theta));
            if (j > bestJ)
            {
                bestJ = j;
                bestIssues = issues;
            }
        }

        // Paiements minimaux (≡ 0) : ils satisfont IR trivialement (pour θ_i = 0,
        // u = 0 - paiement ≥ 0 impose paiement = 0).
        return (new MecanismeGen(bestIssues, new int[4, 2]), bestJ);
    }
}

// Test : profils vrais = tous les agents type 1 (veulent o=1)
var (mecanisme, jAnnonce) = Generateur.Generate(new[] { 1, 1 });

Console.WriteLine("Mécanisme optimal généré :");
Console.WriteLine("  issue_table (issue par profil reporté) :");
for (int r = 0; r < 4; r++)
{
    var types = Domaine.Types(r);
    Console.WriteLine($"    profil reporté=({types[0]}, {types[1]}) -> outcome={mecanisme.Issues[r]}");
}
Console.WriteLine("  payment_table (paiements par profil reporté) :");
for (int r = 0; r < 4; r++)
{
    var types = Domaine.Types(r);
    Console.WriteLine($"    profil reporté=({types[0]}, {types[1]}) -> paiements=({mecanisme.Paiements[r, 0]}, {mecanisme.Paiements[r, 1]})");
}
Console.WriteLine($"Bien-être social J* annoncé = {jAnnonce}");

// --- Assertions exécutables sur les valeurs CITÉES du jumeau Python ---
// DONNÉES CITÉES (source : GameTheory-16b-Automated-Mechanism-Design-Python.ipynb,
// cellule 3) : la table d'issue de M*, ses paiements nuls, et J* = 2. Elles ne
// sont pas recalculées ici : elles confronted le résultat du générateur C#, et
// l'écart est affiché sous la tolérance déclarée.
double tolerance = 1e-9;
int[] issuesCites = { 0, 0, 0, 1 };
int jCite = 2;
int ecartIssues = mecanisme.Issues.Zip(issuesCites, (a, b) => Math.Abs(a - b)).Sum();
int ecartJ = Math.Abs(jAnnonce - jCite);
Console.WriteLine($"Écart sur la table d'issue (somme |diff|) : {ecartIssues} ; sur J* : {ecartJ} (tolérance {tolerance})");
if (ecartIssues > tolerance)
    throw new Exception($"Table d'issue différente de la valeur citée : écart {ecartIssues} > {tolerance}");
if (ecartJ > tolerance)
    throw new Exception($"J* différent de la valeur citée : écart {ecartJ} > {tolerance}");
foreach (int r in Enumerable.Range(0, 4))
    foreach (int i in Enumerable.Range(0, 2))
        if (mecanisme.Paiements[r, i] != 0)
            throw new Exception($"Paiement non nul au profil {r}, agent {i} : la valeur citée les impose nuls");
Console.WriteLine("Assertions exécutables : table d'issue, J* et paiements nuls conformes aux valeurs citées.");


Mécanisme optimal généré :


  issue_table (issue par profil reporté) :


    profil reporté=(0, 0) -> outcome=0


    profil reporté=(0, 1) -> outcome=0


    profil reporté=(1, 0) -> outcome=0


    profil reporté=(1, 1) -> outcome=1


... (4 outputs left out)

    profil reporté=(1, 1) -> paiements=(0, 0)


Bien-être social J* annoncé = 2


Écart sur la table d'issue (somme |diff|) : 0 ; sur J* : 0 (tolérance 1E-09)


Assertions exécutables : table d'issue, J* et paiements nuls conformes aux valeurs citées.


### Exercice 1 — Combien de mécanismes survivent quand on ajoute un agent ?

L'exemple guidé 1 travaille sur `N_AGENTS = 2` : 4 profils de types, 16 tables d'issue. Le domaine est **fixé à deux agents** dans tout le notebook (`N_AGENTS`, `type_domain`, `PROFILES`).

**Protocole** — paiements tous nuls (le cas où IR est satisfaite d'office, toute la contrainte venant de DSIC), pour `n_agents` ∈ {2, 3} :

1. construire la liste des profils de types pour `n` agents ;
2. énumérer **toutes** les tables d'issue (`2^(2^n)` tables) et compter celles qui passent `DSIC` **et** `IR` ;
3. tabuler `(n_agents, nb_profils, nb_tables, nb_admissibles, J* pour le profil tout-1)` ;
4. qualifier : le ratio *admissibles / tables* augmente-t-il ou s'effondre-t-il quand `n` passe de 2 à 3 ?

**Indices**

- À paiements nuls, `IR` est toujours satisfaite (`u_i = θ_i · o ≥ 0`) : compter les admissibles revient à compter les tables **DSIC**.
- Pour un agent de type `θ_i = 1`, la comparaison au report sincère se réduit à `o(r) ≥ o(r̂)` où `r̂` est le profil obtenu en changeant **son** report — la contrainte porte sur une paire de profils voisins.

**Ce que l'exercice fait mesurer** — la taille du domaine explose (4 → 8 profils), mais la part admissible, elle, ne suit pas la même loi : c'est le point que la vue à deux agents ne peut pas montrer.


In [3]:
// Exercice 1 — Combien de mecanismes survivent quand on ajoute un agent ? (a completer)
// TODO etudiant : parametrer le generateur en nombre d'agents et recenser les tables DSIC + IR.
// Etape 1 : profils(n) = produit cartésien de {0, 1} répété n fois (2^n profils).
// Etape 2 : pour n dans N_AGENTS_GRID, enumérer les tables d'issue (paiements tous nuls) et
//           compter celles qui passent DSIC ET IR (paiements nuls -> IR est acquise).
// Etape 3 : tabuler (n, nb_profils, nb_tables, nb_admissibles, J* tout-1) et lire le ratio.
//
// La fonction renvoie null tant qu'elle n'est pas implémentée : le carnet
// s'exécute de bout en bout, l'exercice se remplit ensuite.

int[] N_AGENTS_GRID = { 2, 3 };

Func<int, (int nbProfils, int nbTables, int nbAdmissibles, int jToutUn)?> recensementAdmissibles = nAgents =>
{
    // TODO etudiant : renvoyer (profils, tables d'issue, admissibles DSIC+IR, J* tout-1)
    return null;
};

var reponsesEx1 = N_AGENTS_GRID.Select(n => recensementAdmissibles(n)).ToArray();
if (reponsesEx1.All(x => x is null))
{
    Console.WriteLine("Exercice 1 a completer : generateur parametre en nombre d'agents.");
}
else
{
    // Assertions exécutables : les invariants de STRUCTURE sont vérifiés — ils
    // ne donnent pas la réponse (le compte d'admissibles, lui, reste libre).
    for (int k = 0; k < N_AGENTS_GRID.Length; k++)
    {
        var rep = reponsesEx1[k];
        if (rep is null)
            throw new Exception("Réponse partielle : un résultat par n de la grille est attendu");
        int n = N_AGENTS_GRID[k];
        int nbProfilsAttendu = 1 << n;                 // 2^n profils
        int nbTablesAttendu = 1 << nbProfilsAttendu;   // 2^(2^n) tables d'issue
        if (rep.Value.nbProfils != nbProfilsAttendu)
            throw new Exception($"n={n} : {rep.Value.nbProfils} profils, attendu {nbProfilsAttendu}");
        if (rep.Value.nbTables != nbTablesAttendu)
            throw new Exception($"n={n} : {rep.Value.nbTables} tables d'issue, attendu {nbTablesAttendu}");
        if (rep.Value.nbAdmissibles < 0 || rep.Value.nbAdmissibles > rep.Value.nbTables)
            throw new Exception($"n={n} : nombre d'admissibles hors bornes");
        if (rep.Value.jToutUn < 0)
            throw new Exception($"n={n} : J* tout-1 négatif");
    }
    Console.WriteLine("Exercice 1 : grille remplie ; lire le ratio admissibles / tables quand n passe de 2 à 3.");
}


Exercice 1 a completer : generateur parametre en nombre d'agents.


## Exemple guidé 2 — La vérification (séparée du générateur)

**Énoncé** : implémentez un vérificateur `verify(M, true_types)` qui :

1. Teste **DSIC au sens standard** — le point de départ de la comparaison est le report
   **sincère** `r = (θ₀, θ₁)` (l'autre agent étant sincère), pour chaque agent `i` et chaque
   déviation unilatérale `θ'_i` :
   `u_i(issue_table[(θ₀, θ₁)], θ_i) − payment_i[(θ₀, θ₁)] ≥ u_i(issue_table[(θ'_i, θ_{−i})], θ_i) − payment_i[(θ'_i, θ_{−i})]`

   *Pourquoi cet ancrage* : DSIC garantit que **dire la vérité est optimal** — rien de plus.
   La version initiale de ce vérificateur exigeait l'inégalité depuis **tout** report `r`
   (réserve #12211) : c'est une définition sur-durcie, qui rejette des mécanismes
   DSIC-standard (la cellule suivante en exhibe un témoin). La définition de référence est
   `DSIC` dans [`SocialChoice/social_choice_core_lean/SocialChoice/AMD.lean`](SocialChoice/social_choice_core_lean/SocialChoice/AMD.lean) :
   `u₀ o p₀ x.1 (x.1, x.2.1) ≥ u₀ o p₀ x.1 (x.2.2, x.2.1)` — ancrée au profile sincère.

2. Teste **IR au sens standard** : pour chaque agent `i`, au profile **sincère** uniquement :
   `u_i(issue_table[(θ₀, θ₁)], θ_i) − payment_i[(θ₀, θ₁)] ≥ 0`

3. Teste **`J(M) = J*`** : recalcule `J` sur `true_types` et compare à la valeur annoncée par le générateur.

**Contrainte de génie logiciel** : le vérificateur **ne doit pas importer le générateur**. Il prend
`M = (issue_table, payment_table)` en argument et n'utilise que la définition d'`utility_i` et le calcul
de `J` qu'il ré-implémente localement. C'est exactement la **Loi II** du chantier : générateur ≠ vérificateur.

**Version C#** : la séparation prend une forme plus forte encore. Le vérificateur déclare **son propre
type** de mécanisme (`MecanismeVerifie`) et **ses propres** primitives ; il ne référence ni le type
`MecanismeGen`, ni la classe `Generateur`, ni aucune de leurs méthodes. Il reconstruit même son
propre `M*` par une énumération locale : ce qu'il vérifie n'est pas la sortie du générateur, c'est le
mécanisme tel que le vérificateur le recompose. Une séparation qui tient par le typage ne peut pas
se défaire par un appel oublié.



In [4]:
// Cellule 5 — Vérificateur SÉPARÉ (n'importe pas le générateur)
//
// Calibré sur la définition STANDARD (tranche 2, réserve #12211) : DSIC et IR ancrent
// la comparaison au report SINCÈRE — même sens que `DSIC`/`IR` dans
// SocialChoice/social_choice_core_lean/SocialChoice/AMD.lean (la référence formelle). Les versions
// sur-durcies initiales (inégalité depuis TOUT report r) sont conservées sous un
// nom explicite pour exhiber la divergence (témoin : la dictature, ci-dessous).
//
// SÉPARATION, version C# : le vérificateur déclare SON PROPRE type de mécanisme
// (`MecanismeVerifie`) et redéfinit TOUTES ses primitives — il ne référence ni
// `MecanismeGen`, ni `Generateur`, ni aucune méthode du générateur, et il
// reconstruit son propre M* par une énumération locale.

record MecanismeVerifie(int[] Issues, int[,] Paiements);

static class Verificateur
{
    // Copie locale de l'utilité — n'utilise pas Domaine.UtilityI.
    static int UtilityI(int i, int outcome, int thetaI) => thetaI * outcome;

    // Index d'un profil à partir des types reportés (convention locale).
    static int Index(int[] types) => 2 * types[0] + types[1];

    // Les 4 profils de types, dans l'ordre du produit cartésien.
    static int[][] Profiles() => new[] { new[] { 0, 0 }, new[] { 0, 1 }, new[] { 1, 0 }, new[] { 1, 1 } };

    // Les 16 tables d'issue, dans l'ordre du produit cartésien.
    static int[] Issues(int masque) => Enumerable.Range(0, 4).Select(r => (masque >> (3 - r)) & 1).ToArray();

    // J recalculé localement.
    public static int SocialWelfare(int[] issueTable, int[] trueTypes)
    {
        int issue = issueTable[Index(trueTypes)];
        return trueTypes.Sum(theta => UtilityI(0, issue, theta));
    }

    // DSIC au sens STANDARD : le point de départ est le report sincère
    // (θ₀, θ₁) — l'autre agent étant sincère. Pour chaque agent i et chaque
    // déviation unilatérale θ', l'utilité du report sincère domine.
    public static (bool ok, string message) VerifyDsic(MecanismeVerifie m, int[] trueTypes)
    {
        int sincere = Index(trueTypes);
        for (int i = 0; i < 2; i++)
        {
            int thetaI = trueTypes[i];
            double uSincere = UtilityI(i, m.Issues[sincere], thetaI) - m.Paiements[sincere, i];
            for (int thetaDev = 0; thetaDev < 2; thetaDev++)
            {
                int[] typesDev = (int[])trueTypes.Clone();
                typesDev[i] = thetaDev;
                int dev = Index(typesDev);
                double uDeviation = UtilityI(i, m.Issues[dev], thetaI) - m.Paiements[dev, i];
                if (uDeviation > uSincere + 1e-9)
                    return (false, $"DSIC fail: agent {i} préfère r'=({typesDev[0]}, {typesDev[1]}) au sincère ({trueTypes[0]}, {trueTypes[1]})");
            }
        }
        return (true, "DSIC OK (ancrage au report sincère)");
    }

    // Variante SUR-DURCIE (l'ancienne) : exige l'inégalité depuis tout report r,
    // pas seulement le sincère. Conservée pour la divergence, pas utilisée comme
    // critère d'admissibilité (réserve #12211).
    public static (bool ok, string message) VerifyDsicToutReport(MecanismeVerifie m, int[] trueTypes)
    {
        foreach (int[] r in Profiles())
        {
            for (int i = 0; i < 2; i++)
            {
                int thetaI = trueTypes[i];
                double uTruthful = UtilityI(i, m.Issues[Index(r)], thetaI) - m.Paiements[Index(r), i];
                for (int thetaDev = 0; thetaDev < 2; thetaDev++)
                {
                    int[] typesDev = (int[])r.Clone();
                    typesDev[i] = thetaDev;
                    int dev = Index(typesDev);
                    double uDeviation = UtilityI(i, m.Issues[dev], thetaI) - m.Paiements[dev, i];
                    if (uDeviation > uTruthful + 1e-9)
                        return (false, $"sur-durci: agent {i} préfère r'=({typesDev[0]}, {typesDev[1]}) à r=({r[0]}, {r[1]}) (départ NON sincère)");
                }
            }
        }
        return (true, "sur-durci OK");
    }

    // IR au sens STANDARD : l'utilité du report SINCÈRE est non négative,
    // pour chaque agent.
    public static (bool ok, string message) VerifyIr(MecanismeVerifie m, int[] trueTypes)
    {
        int sincere = Index(trueTypes);
        for (int i = 0; i < 2; i++)
        {
            double u = UtilityI(i, m.Issues[sincere], trueTypes[i]) - m.Paiements[sincere, i];
            if (u < -1e-9)
                return (false, $"IR fail: agent {i} u={u} au profile sincère ({trueTypes[0]}, {trueTypes[1]})");
        }
        return (true, "IR OK (au report sincère)");
    }

    // Variante SUR-DURCIE (l'ancienne) : utilité ≥ 0 à TOUT report.
    public static (bool ok, string message) VerifyIrToutReport(MecanismeVerifie m, int[] trueTypes)
    {
        foreach (int[] r in Profiles())
        {
            for (int i = 0; i < 2; i++)
            {
                double u = UtilityI(i, m.Issues[Index(r)], trueTypes[i]) - m.Paiements[Index(r), i];
                if (u < -1e-9)
                    return (false, $"sur-durci IR: agent {i} u={u} en profile ({r[0]}, {r[1]})");
            }
        }
        return (true, "sur-durci IR OK");
    }

    // J(M) = J annoncé ?
    public static (bool ok, string message) VerifyJ(MecanismeVerifie m, int[] trueTypes, int jAnnonce)
    {
        int jComputed = SocialWelfare(m.Issues, trueTypes);
        if (Math.Abs(jComputed - jAnnonce) > 1e-9)
            return (false, $"J computed={jComputed} ≠ announced={jAnnonce}");
        return (true, $"J OK ({jComputed})");
    }

    // Les trois vérifications, séparément.
    public static List<(string nom, bool ok, string message)> Verify(MecanismeVerifie m, int[] trueTypes, int jAnnonce)
    {
        var resultats = new List<(string, bool, string)>();
        var (okDs, msgDs) = VerifyDsic(m, trueTypes);
        resultats.Add(("DSIC", okDs, msgDs));
        var (okIr, msgIr) = VerifyIr(m, trueTypes);
        resultats.Add(("IR", okIr, msgIr));
        var (okJ, msgJ) = VerifyJ(m, trueTypes, jAnnonce);
        resultats.Add(("J", okJ, msgJ));
        return resultats;
    }

    // Reconstruction locale de M* : le vérificateur ne lit pas la sortie du
    // générateur, il recompose son propre mécanisme par énumération.
    public static (MecanismeVerifie mecanisme, int bienEtre) TestGenerate(int[] trueTypes)
    {
        int bestJ = -1;
        int[] bestIssues = Issues(0);
        for (int masque = 0; masque < 16; masque++)
        {
            int[] issues = Issues(masque);
            int j = SocialWelfare(issues, trueTypes);
            if (j > bestJ)
            {
                bestJ = j;
                bestIssues = issues;
            }
        }
        return (new MecanismeVerifie(bestIssues, new int[4, 2]), bestJ);
    }
}

// --- Test 1 : M* reconstruit par le vérificateur (true_types = [1, 1]) ---
var (mStar, jStar) = Verificateur.TestGenerate(new[] { 1, 1 });
Console.WriteLine("Vérification de M* sur true_types = [1, 1] (définitions standard) :");
foreach (var (nom, ok, message) in Verificateur.Verify(mStar, new[] { 1, 1 }, jStar))
    Console.WriteLine($"  {(ok ? "✓" : "✗")} {nom} : {message}");
var (okSur, msgSur) = Verificateur.VerifyDsicToutReport(mStar, new[] { 1, 1 });
Console.WriteLine($"  ({(okSur ? "✓" : "✗")} variante sur-durcie : {msgSur})");

// --- Assertions exécutables sur les messages CITÉS du jumeau Python ---
// DONNÉES CITÉES (source : jumeau Python, cellule 5) : le message exact de la
// variante sur-durcie sur M*, et le verdict du croisement sur le profil (0, 0).
string msgSurCite = "sur-durci: agent 0 préfère r'=(1, 1) à r=(0, 1) (départ NON sincère)";
if (okSur) throw new Exception("La variante sur-durcie doit échouer sur M* (valeur citée)");
if (msgSur != msgSurCite)
    throw new Exception($"Message sur-durcie différent de la valeur citée : « {msgSur} »");
Console.WriteLine($"Message sur-durcie conforme à la valeur citée : « {msgSur} »");

Console.WriteLine();
Console.WriteLine("Vérification croisée : un autre profile (true_types = [0, 0]) :");
// M* a payments=0 partout, donc u = 0*outcome = 0 -> IR OK trivialement
// DSIC : si agent type 0 devie vers θ=1, outcome change (selon issue_table[profile_vrai_modifie]).
// Mais θ_i = 0 -> utility_i = 0 peu importe outcome -> DSIC OK.
foreach (var (nom, ok, message) in Verificateur.Verify(mStar, new[] { 0, 0 }, jStar))
    Console.WriteLine($"  {(ok ? "✓" : "✗")} {nom} : {message}");

var (okJ00, msgJ00) = Verificateur.VerifyJ(mStar, new[] { 0, 0 }, jStar);
string msgJCite = "J computed=0 ≠ announced=2";
if (okJ00) throw new Exception("Le croisement sur (0, 0) doit recalculer J = 0 (valeur citée)");
if (msgJ00 != msgJCite)
    throw new Exception($"Message de J différent de la valeur citée : « {msgJ00} »");

// --- Test 2 : TÉMOIN DE DIVERGENCE — la dictature de l'agent 1 ---
//
// issue = r₁ (l'issue suit exactement le report de l'agent 1), paiements nuls.
//   DSIC-standard  : PASS — le dictateur n'a rien à gagner à mentir (l'issue suit
//                    son report) ; l'agent 0 n'influence pas l'issue.
//   Sur-durci      : FAIL — au report (1, 0) avec types vrais [1, 1], l'agent 1
//                    « devie » vers (1, 1) et gagne 1. Mais ce « gain » est la
//                    CORRECTION d'un mensonge : DSIC-standard ne garantit que
//                    « dire la vérité est optimal », pas « toute position de
//                    départ est indeviablable ». La variante sur-durcie confond
//                    corriger son mensonge et manipuler — d'où la réserve #12211.
Console.WriteLine();
Console.WriteLine("TÉMOIN DE DIVERGENCE — la dictature de l'agent 1 (issue = r₁, paiements nuls) :");
var mDict = new MecanismeVerifie(new[] { 0, 1, 0, 1 }, new int[4, 2]);
var (okStd, msgStd) = Verificateur.VerifyDsic(mDict, new[] { 1, 1 });
var (okDur, msgDur) = Verificateur.VerifyDsicToutReport(mDict, new[] { 1, 1 });
Console.WriteLine($"  DSIC standard  : {(okStd ? "✓" : "✗")} — {msgStd}");
Console.WriteLine($"  DSIC sur-durci : {(okDur ? "✓" : "✗")} — {msgDur}");

// DONNÉE CITÉE (jumeau Python, cellule 5) : le message de la variante sur-durcie
// sur le témoin de dictature.
string msgDurCite = "sur-durci: agent 1 préfère r'=(0, 1) à r=(0, 0) (départ NON sincère)";
if (!okStd) throw new Exception("La définition standard doit valider le témoin de dictature");
if (okDur) throw new Exception("La variante sur-durcie doit rejeter le témoin de dictature");
if (msgDur != msgDurCite)
    throw new Exception($"Message sur-durcie du témoin différent de la valeur citée : « {msgDur} »");
Console.WriteLine("Assertions exécutables : DSIC standard ✓, sur-durcie ✗, messages conformes aux valeurs citées.");


Vérification de M* sur true_types = [1, 1] (définitions standard) :


  ✓ DSIC : DSIC OK (ancrage au report sincère)


  ✓ IR : IR OK (au report sincère)


  ✓ J : J OK (2)


  (✗ variante sur-durcie : sur-durci: agent 0 préfère r'=(1, 1) à r=(0, 1) (départ NON sincère))


Message sur-durcie conforme à la valeur citée : « sur-durci: agent 0 préfère r'=(1, 1) à r=(0, 1) (départ NON sincère) »


... (4 outputs left out)

TÉMOIN DE DIVERGENCE — la dictature de l'agent 1 (issue = r₁, paiements nuls) :


  DSIC standard  : ✓ — DSIC OK (ancrage au report sincère)


  DSIC sur-durci : ✗ — sur-durci: agent 1 préfère r'=(0, 1) à r=(0, 0) (départ NON sincère)


Assertions exécutables : DSIC standard ✓, sur-durcie ✗, messages conformes aux valeurs citées.


### Lecture du résultat — ce que le calibrage change

Le vérificateur, dans sa version initiale, exigeait l'inégalité d'incitation **depuis tout report `r`**
— y compris des reports mensongers. C'est strictement plus fort que DSIC : le témoin de la cellule
précédente (la **dictature de l'agent 1**) passe la définition standard et échoue à la sur-durcie.
La divergence n'est pas académique : un générateur filtré par la version sur-durcie éliminerait des
mécanismes admissibles au sens de Sandholm — l'espace de recherche serait amputé sans gain théorique,
puisque DSIC ne promet que *« dire la vérité est optimal »*, jamais *« aucune position de départ n'est
améliorable »*.

Les deux définitions **divergent jusque sur M\*** : dans le jumeau Python, la sortie d'avant calibrage
montrait `✗ DSIC : agent 0 préfère r'=(1, 1) à r=(0, 1)` — une **contradiction frontale avec le
certificat Lean** `amd_star_DSIC` (PR #12648), qui prouve que M\* est DSIC sur tout le domaine. Ce
n'était pas un bug du générateur : c'est la définition sur-durcie qui voyait une « manipulation » dans la
correction d'un mensonge. Le calibrage résout la contradiction — Python et Lean disent désormais
la même chose sur M\*.
La référence formelle est `DSIC`/`IR` dans
[`AMD.lean`](SocialChoice/social_choice_core_lean/SocialChoice/AMD.lean) (PR #12648) : les définitions Lean ancrent
la comparaison au profile sincère, et ce vérificateur est désormais calibré sur elles — le
vérificateur du carnet et son compagnon formel disent la même chose.

**Ce que le portage C# ajoute** : la séparation ne repose plus sur une discipline de relecture, mais
sur le typage. Le vérificateur C# ne peut pas appeler le générateur — il ne connaît ni sa classe, ni
son type de mécanisme, ni ses méthodes. La garantie est structurelle, et non éditoriale.



### Exercice 2 — Vérifier M\* sur **tous** les profils, pas seulement ceux de l'exemple

L'exemple guidé 2 vérifie `M\*` sur `[1, 1]`, puis fait une **vérification croisée** sur `[0, 0]` — où `J` recalcule 0 alors que `J\* = 2` était annoncé. La cellule s'arrête là : deux profils sur quatre, et le constat n'est pas généralisé.

**Protocole** — `M\*` de l'exemple guidé 1 (outcome 1 si et seulement si le report est `(1, 1)`, paiements nuls), pour les **quatre** profils vrais `PROFILES` :

1. calculer `J(M\*) = Σ_i θ_i · issue_table[(θ₀, θ₁)]` ;
2. calculer `J\*(profil)`, l'optimum de ce profil, par énumération des tables d'issue admissibles (paiements nuls) ;
3. tabuler `(profil vrai, J(M\*), J\*(profil), perte)` et **nommer les profils où `M\*` est optimal** ;
4. conclure : que dit la perte sur ce que « `M\*` » désigne réellement ?

**Indices**

- `J(M\*)` ne dépend que du profil **sincère** : `issue_table[(θ₀, θ₁)]` est lu une seule fois par profil.
- La perte à un profil mixte est le fait marquant : elle n'est pas un défaut du générateur, elle dit **pour quelle instance** le mécanisme a été calculé.

**Ce que l'exercice fait mesurer** — un mécanisme AMD est une réponse à **une instance** ; la vérification à deux profils de l'exemple guidé 2 laissait croire à une portée plus large.


In [5]:
// Exercice 2 — Verifier M* sur TOUS les profils (a completer)
// TODO etudiant : mesurer J(M*) profil par profil et le comparer a l'optimum de ce profil.
// Etape 1 : reconstruire M* de l'exemple guide 1 (issue=1 ssi report=(1,1), paiements nuls).
// Etape 2 : pour chaque profil vrai, J(M*) = sum_i theta_i * issue_table[profil].
// Etape 3 : enumerer les tables d'issue admissibles (paiements nuls) pour obtenir J*(profil),
//           puis tabuler (profil, J(M*), J*(profil), perte) et nommer les profils ou M* est optimal.
//
// La fonction renvoie null tant qu'elle n'est pas implémentée : le carnet
// s'exécute de bout en bout, l'exercice se remplit ensuite.

Func<(int[] profil, int jMstar, int jStarProfil, int perte)[]> perteParProfil = () => null;

var reponsesEx2 = perteParProfil();
if (reponsesEx2 is null)
{
    Console.WriteLine("Exercice 2 a completer : verification de M* sur tous les profils.");
}
else
{
    // Assertions exécutables : la STRUCTURE de la table et les valeurs DÉJÀ
    // produites par l'exemple guidé 1 sont vérifiées ; l'optimum J*(profil)
    // reste libre — c'est précisément l'objet de l'exercice.
    if (reponsesEx2.Length != 4)
        throw new Exception($"Un profil par type-vrai est attendu : {reponsesEx2.Length} lignes");
    for (int k = 0; k < reponsesEx2.Length; k++)
    {
        var (profil, jMstar, jStarProfil, perte) = reponsesEx2[k];
        if (profil is null || profil.Length != 2)
            throw new Exception("Profil mal formé");
        // La table d'issue de M* est celle de l'exemple guidé 1 : issue 1 si et
        // seulement si le profil reporté est (1, 1).
        int issueCite = (profil[0] == 1 && profil[1] == 1) ? 1 : 0;
        int jMstarAttendu = (profil[0] + profil[1]) * issueCite;
        if (jMstar != jMstarAttendu)
            throw new Exception($"J(M*) faux au profil ({profil[0]}, {profil[1]}) : {jMstar} au lieu de {jMstarAttendu}");
        if (jStarProfil < jMstar)
            throw new Exception("L'optimum du profil ne peut pas être inférieur à J(M*)");
        if (perte != jStarProfil - jMstar)
            throw new Exception("La perte est J*(profil) - J(M*)");
    }
    Console.WriteLine("Exercice 2 : table des quatre profils remplie ; nommer les profils où M* est optimal.");
}


Exercice 2 a completer : verification de M* sur tous les profils.


## Exemple guidé 3 — L'impossibilité (avec témoin)

**Énoncé** : durcissez les contraintes jusqu'à ce qu'aucun mécanisme admissible n'existe. La sortie obligatoire est **soit** :

- un **témoin d'impossibilité** (preuve qu'aucun mécanisme ne satisfait toutes les contraintes), **soit**
- pour chaque mécanisme candidat, une **déviation profitable exhibée** (l'agent qui dévie, le profile, le gain).

Un échec sans témoin n'est pas un résultat.

Cas : 2 agents, chacun a 2 types (θ ∈ {0, 1}), 1 issue binaire. `J = sum_i θ_i * o`. Contraintes : DSIC + IR + **paiement strictement positif** (`payment_i > 0` pour CHAQUE profile reporté, pour CHAQUE agent).

Pour ce cas, montrez que **les paiements strictement positifs violent IR** pour les agents de type 0 (utilité = 0 - payment < 0) — donc l'ensemble des candidats admissibles est VIDE.



In [6]:
// Cellule 7 — Témoin d'impossibilité : paiement strictement positif viole IR pour type=0
//
// L'énumération est locale (comme dans la source Python) ; le CONTRÔLE d'IR,
// lui, est celui du vérificateur de la cellule 5 : on ne redérive pas IR à la
// main, on réutilise l'organe natif du carnet.

// Énumère tous les mécanismes dont CHAQUE paiement est ≥ 1.
static IEnumerable<MecanismeVerifie> EnumererPaiementsStrictementPositifs()
{
    // Table d'issue : les 16 tables du domaine, dans l'ordre du produit cartésien.
    for (int masqueIssues = 0; masqueIssues < 16; masqueIssues++)
    {
        var issues = Enumerable.Range(0, 4).Select(r => (masqueIssues >> (3 - r)) & 1).ToArray();
        // Paiements : 4 profils × 2 agents, chacun dans {1, 2} (strictement positif).
        // L'ordre du produit cartésien est respecté : la dernière coordonnée
        // varie le plus vite.
        for (int numeroPaiements = 0; numeroPaiements < 256; numeroPaiements++)
        {
            var paiements = new int[4, 2];
            int reste = numeroPaiements;
            for (int k = 7; k >= 0; k--)
            {
                paiements[k / 2, k % 2] = 1 + reste % 2;   // {1, 2}
                reste /= 2;
            }
            yield return new MecanismeVerifie(issues, paiements);
        }
    }
}

// IR standard à TOUS les profils sincères — réutilise le vérificateur (cellule 5),
// l'organe natif du notebook, au lieu de re-dériver IR à la main.
static bool IrAdmissible(MecanismeVerifie m)
{
    int[][] profils = { new[] { 0, 0 }, new[] { 0, 1 }, new[] { 1, 0 }, new[] { 1, 1 } };
    return profils.All(r => Verificateur.VerifyIr(m, r).ok);
}

// Le compte de 4096 devient une preuve : on ne se contente plus de compter les
// candidats, on vérifie IR sur chacun. Pour un agent de type θ_i = 0,
// u_i = 0·outcome − payment_i = −payment_i ≤ −1 < 0 : IR est violé. Le calcul
// ci-dessous le confirme sur les 4096 candidats (0 IR-admissible).
int nCandidats = 0;
int nIrOk = 0;
foreach (var m in EnumererPaiementsStrictementPositifs())
{
    nCandidats++;
    if (IrAdmissible(m))
        nIrOk++;
}
Console.WriteLine($"Candidats avec paiement strictement positif : {nCandidats}");
Console.WriteLine($"Dont IR-admissibles (IR ok à TOUS profils) : {nIrOk}");
Console.WriteLine($"IR violée à au moins un profil : {nCandidats - nIrOk} / {nCandidats}");

// --- Assertions exécutables sur les valeurs CITÉES du jumeau Python ---
// DONNÉES CITÉES (source : jumeau Python, cellule 7) : 4096 candidats, 0
// IR-admissible, 4096/4096 avec IR violée.
int candidatsCites = 4096;
int irOkCite = 0;
if (nCandidats != candidatsCites)
    throw new Exception($"Nombre de candidats différent de la valeur citée : {nCandidats} au lieu de {candidatsCites}");
if (nIrOk != irOkCite)
    throw new Exception($"Nombre d'IR-admissibles différent de la valeur citée : {nIrOk} au lieu de {irOkCite}");
if (nCandidats - nIrOk != candidatsCites)
    throw new Exception("Le compte des violations doit couvrir tous les candidats");

// Témoin EXPLICITE, sur le premier candidat : l'agent de type 0 au profil (0, 0)
// a une utilité strictement négative — c'est le témoin d'impossibilité.
var premierCandidat = EnumererPaiementsStrictementPositifs().First();
double uTemoin = 0 * premierCandidat.Issues[0] - premierCandidat.Paiements[0, 0];
Console.WriteLine($"Témoin : agent 0 de type 0 au profil (0, 0) a u = {uTemoin} < 0 -> IR violée");
if (!(uTemoin < 0))
    throw new Exception("Le témoin doit exhiber une utilité strictement négative");
Console.WriteLine("Assertions exécutables : compte de candidats, compte d'IR-admissibles et témoin tous conformes.");


Candidats avec paiement strictement positif : 4096


Dont IR-admissibles (IR ok à TOUS profils) : 0


IR violée à au moins un profil : 4096 / 4096


Témoin : agent 0 de type 0 au profil (0, 0) a u = -1 < 0 -> IR violée


Assertions exécutables : compte de candidats, compte d'IR-admissibles et témoin tous conformes.


**Lecture du témoin d'impossibilité.** Pour chacun des candidats énumérés ci-dessus, l'agent de type θ_i = 0 rapportant le profile r obtient u_i = 0 · outcome(r) − payment_i(r) = −payment_i(r) ≤ −1 < 0 : IR est violé.

- **Contraintes** : DSIC ∧ IR ∧ (∀i, ∀r : payment_i(r) ≥ 1).
- **Preuve** : pour un agent de type θ_i = 0 reportant un profile r : u_i = θ_i · outcome(r) − payment_i(r) = −payment_i(r) ≤ −1 (car payment_i(r) ≥ 1) → IR violé (u_i < 0).
- **Conséquence** : pour tout candidat, l'agent de type 0 a une utilité strictement négative en rapportant truthful. Aucun mécanisme avec paiement ≥ 1 ne satisfait IR pour tous les profiles.
- **Conclusion** : l'ensemble des mécanismes admissibles est **vide**. Le témoin est la construction explicite — pour chaque candidat : agent i, profile r, déviation vers truthful donne u < 0.



### Exercice 3 — Plafond de paiement ou plancher de surplus : lequel vide l'ensemble ?

L'exemple guidé 3 durcit les contraintes par un **plancher sur les paiements** (`payment_i ≥ 1`) et conclut que l'ensemble admissible est vide. La question laissée ouverte : cette impossibilité vient-elle du fait qu'on **borne les paiements**, ou du fait qu'on **exige une utilité garantie** ?

**Protocole** — domaine déclaré, paiements ∈ {0, 1} (4 096 candidats : 16 tables d'issue × 2⁸ tables de paiement) ; deux durcissements à comparer :

- **A — plafond** : tous les paiements nuls (`∀i, ∀r : p_i(r) = 0`) sous `DSIC ∧ IR` ;
- **B — plancher de surplus** : `∀i, ∀r`, au report sincère, `u_i ≥ 1`, sous `DSIC`.

1. compter les candidats admissibles `DSIC ∧ IR` de l'espace entier (référence) ;
2. compter les admissibles de **A**, puis de **B** ;
3. du côté vidé, exhiber le **témoin par candidat** : l'agent, le profil et l'utilité qui violent la contrainte ;
4. conclure : lequel des deux durcissements porte l'impossibilité, et pourquoi ?

**Indices**

- Un agent de type `θ_i = 0` a `u_i = 0 · o − p_i = −p_i` : chercher ce que cela implique pour un **plancher** de surplus, et ce que cela laisse possible pour un **plafond**.
- Un plafond ne fait que restreindre l'espace **par le haut** ; il laisse `p ≡ 0` dans le jeu de candidats — vérifier que ce candidat est bien admissible.

**Ce que l'exercice fait mesurer** — l'impossibilité de l'exemple guidé 3 n'est pas due au plafonnement des paiements mais à la contrainte **IR** durcie : séparer les deux est ce qui rend la conclusion transposable.


In [7]:
// Exercice 3 — Plafond de paiement ou plancher de surplus ? (a completer)
// TODO etudiant : comparer un PLAFOND de paiement et un PLANCHER de surplus garanti.
// Etape 1 : enumerer l'espace paiements in {0,1} et compter les admissibles DSIC + IR (reference).
// Etape 2 : durcissement A = plafond (tous paiements nuls) -> compter les admissibles.
//           durcissement B = plancher de surplus (u_i >= 1 au report sincere) -> compter les admissibles.
// Etape 3 : du cote vide, exhiber un temoin par candidat (agent, profil, utilite) et conclure.
//
// La fonction renvoie null tant qu'elle n'est pas implémentée : le carnet
// s'exécute de bout en bout, l'exercice se remplit ensuite.

int plafond = 0;
int surplusMin = 1;

Func<string, (int nbCandidats, int nbAdmissibles, int nbTemoins)?> censusDurci = mode =>
{
    // TODO etudiant : renvoyer (candidats, admissibles, témoins) pour mode = "plafond" ou "surplus"
    return null;
};

var reponsesEx3 = new[] { "plafond", "surplus" }.Select(mode => censusDurci(mode)).ToArray();
if (reponsesEx3.All(x => x is null))
{
    Console.WriteLine("Exercice 3 a completer : plafond de paiement contre plancher de surplus.");
}
else
{
    // Assertions exécutables : la TAILLE de l'espace et l'articulation entre
    // admissibles et témoins sont vérifiées ; les deux comptes d'admissibles,
    // eux, restent libres — ils sont l'objet de l'exercice.
    for (int k = 0; k < reponsesEx3.Length; k++)
    {
        var rep = reponsesEx3[k];
        if (rep is null)
            throw new Exception("Réponse partielle : les deux durcissements sont attendus");
        // DONNÉE CITÉE (jumeau Python, cellule 14) : l'espace compte 4096
        // candidats (16 tables d'issue × 2^8 tables de paiement).
        if (rep.Value.nbCandidats != 4096)
            throw new Exception($"Espace de candidats inattendu : {rep.Value.nbCandidats}");
        if (rep.Value.nbAdmissibles < 0 || rep.Value.nbAdmissibles > rep.Value.nbCandidats)
            throw new Exception("Nombre d'admissibles hors bornes");
        if (rep.Value.nbTemoins != rep.Value.nbCandidats - rep.Value.nbAdmissibles)
            throw new Exception("Le nombre de témoins est le complément du nombre d'admissibles");
    }
    Console.WriteLine("Exercice 3 : les deux durcissements sont comptés ; conclure sur celui qui vide l'ensemble.");
}


Exercice 3 a completer : plafond de paiement contre plancher de surplus.


## À retenir

L'AMD (Conitzer–Sandholm) optimise **dans un espace de mécanismes DONNÉ** : on a fixé Θ, O, U, J, les contraintes — le solveur cherche le meilleur `M ∈ ℳ`. C'est un bond énorme, mais c'est un bond **dans l'espace**.

La **strate 7** demande autre chose : **comment apparaît une coordonnée qui n'appartenait pas encore à cet espace ?** Comment une nouvelle dimension d'utilité, un nouveau type d'agent, un nouveau canal de communication entre-t-il dans `ℳ` ? Le notebook `GameTheory-03d-Deux-Especes-de-Fleches-Python` traite précisément ce point : les flèches comme témoins d'une coordonnée qui s'invente dans le jeu.

**Sandholm nous emmène très loin dans la strate 7, mais il ne nous dispense pas de la strate 7.** Le générateur AMD reste un solveur dans un cadre déjà arrêté — son apport à la digestion tient à la **discipline de la table explicite** et du **vérificateur séparé**, pas à l'invention de la coordonnée manquante.

***

## Récapitulatif des trois exemples guidés

| Exercice | Sortie | Vérification |
|----------|--------|--------------|
| Exemple guidé 1 — Générateur | `M* = (issue_table, payment_table)` exhibée | `J(M*)` recalculé |
| Exemple guidé 2 — Vérificateur séparé | `DSIC`, `IR`, `J` (3 résultats indépendants) | Le vérificateur n'importe pas le générateur |
| Exemple guidé 3 — Témoin d'impossibilité | Déviation profitable exhibée pour chaque candidat (ici : type=0, u<0) | L'énumération des paiements strictement positifs ne retient aucun admissible |

***

## Suite du chantier (B3, #12205)

Le **compagnon Lean** est livré : [`AMD.lean`](SocialChoice/social_choice_core_lean/SocialChoice/AMD.lean) (PR #12648)
certifie M\* DSIC/IR/J\*=2/optimal au sens standard, plus l'impossibilité générale « paiement
strictement positif ⇒ IR violé » (miroir de l'exemple guidé 3). La présente tranche a calibré le
vérificateur sur ces mêmes définitions (réserve #12211 soldée) — générateur, vérificateur et
compagnon formel parlent désormais la même langue.

***

**Refs** : #12211 · Sandholm 2002 (AMDA) · Conitzer & Sandholm 2003 · Vervaeke #11488


**Trois mesures à retenir.**

- Le générateur exhibe `M*` : issue 1 au profil `(1, 1)`, issue 0 partout ailleurs, **paiements nuls partout**, `J*` annoncé à 2. Le vérificateur séparé confirme `DSIC`, `IR` et `J = 2` au profil `(1, 1)`.
- Au profil `(0, 0)`, le même vérificateur recalcule `J = 0` : l'annonce `J* = 2` vaut **au profil de référence**, pas pour toute la table.
- Le témoin d'impossibilité enumère **4096** candidats à paiement strictement positif et n'en retient **aucun** admissible : pour un agent de type 0, `u = -payment ≤ -1 < 0`, donc IR est violée par construction.



## Vérifiez votre compréhension

1. Le vérificateur valide `DSIC`, `IR` et `J = 2` au profil `(1, 1)`, mais recalcule `J = 0` au profil `(0, 0)`. Le générateur a-t-il produit une table fausse ?

   <details><summary>Réponse</summary>

   Non : il a produit une table qui n'est **pas** constante en bien-être, et le vérificateur a fait son travail. `J* = 2` est une promesse faite **au profil de référence** `(1, 1)` — l'issue 1 y vaut 2, l'issue 0 partout ailleurs. La vérification croisée ne mesure donc pas une contradiction du générateur mais la **portée** de son annonce : `J*` vaut 2 si et seulement si les vrais types sont `(1, 1)`. C'est précisément l'intérêt d'un vérificateur qui n'importe pas le générateur : il peut le contredire, y compris sur un point que le générateur ne prétendait pas couvrir.

   </details>

2. La variante « sur-durcie » de l'inégalité d'incitation exige que l'inégalité tienne **depuis tout report**, y compris mensongers. Elle échoue même sur `M*`. `M*` est-il non incitatif ?

   <details><summary>Réponse</summary>

   Non : l'échec porte sur la définition, pas sur le mécanisme. DSIC promet que « dire la vérité est optimal » ; la sur-durcie promettait « aucune position de départ n'est améliorable », ce qui est strictement plus fort et amputerait l'espace de recherche de mécanismes admissibles au sens de Sandholm. La sur-durcie signalait une « manipulation » dans la **correction** d'un mensonge : l'agent 0 préfère `r' = (1, 1)` à `r = (0, 1)`, ce que DSIC autorise explicitement. Le calibrage a réaligné le vérificateur sur les définitions Lean de `AMD.lean` (`amd_star_DSIC`, PR #12648) : les deux langages disent désormais la même chose sur `M*`.

   </details>

3. Le témoin d'impossibilité porte sur les candidats à paiement strictement positif. Pourquoi l'agent de type 0 suffit-il à en finir avec **tous** les candidats ?

   <details><summary>Réponse</summary>

   Parce que la contrainte porte sur **tout** profil, et qu'un agent de type 0 existe sur tout profil. Pour un tel agent, `u = θ·outcome(r) − payment(r) = −payment(r) ≤ −1 < 0` dès qu'on lui verse au moins 1 : l'utilité du report sincère est négative, donc IR est violée. La preuve est **point-prior** — elle porte sur le profil `(1, 1)` — et c'est ce point-prior qui la rend applicable à tout candidat de l'espace : l'ensemble admissible n'est pas « petit », il est **vide**.

   </details>

4. Le carnet annonce que le générateur explore un espace **borné** de mécanismes. Qu'est-ce que l'AMD ne règle pas, selon la conclusion ?

   <details><summary>Réponse</summary>

   La **strate 7** : l'AMD optimise dans un espace de mécanismes *donné* — `Θ`, `O`, `U`, `J` et les contraintes sont arrêtés. Il ne répond pas à la question « comment une coordonnée qui n'appartenait pas encore à cet espace apparaît-elle ? » : nouvelle dimension d'utilité, nouveau type d'agent, nouveau canal de communication. Sandholm emmène très loin dans cette direction sans y dispenser. L'apport du carnet à la digestion tient à la **discipline de la table explicite** et à la **séparation générateur / vérificateur**, pas à l'invention de la coordonnée manquante — c'est le sens du renvoi vers `GameTheory-03d-Deux-Especes-de-Fleches-Python`.

   </details>



## Pour aller plus loin

Les références ci-dessous ont été **vérifiées depuis Crossref** (DOI confirmé via `api.crossref.org/works/{doi}`) à la date de la livraison. Le mapping section par section suit la progression du notebook : introduction à AMD, complexité du générateur, calibration du vérificateur, et sandbox large-scale pour l'impossibilité.

1. **Conitzer V. & Sandholm T. (2003)** — *Automated mechanism design for a self-interested designer*. EC '03: Proceedings of the 4th ACM conference on Electronic commerce, San Diego, CA, USA, 232–233. DOI: [10.1145/779928.779974](https://doi.org/10.1145/779928.779974) — §1 (papier fondateur AMD : le designer est lui-même stratégique, ce qui renverse la position classique du design de mécanismes).

2. **Conitzer V. & Sandholm T. (2003)** — *Automated mechanism design: complexity results stemming from the single-agent setting*. ICEC '03: Proceedings of the 5th international conference on Electronic commerce, Pittsburgh, PA, USA, 17–24. DOI: [10.1145/948005.948008](https://doi.org/10.1145/948005.948008) — §2 (l'AMD single-agent est déjà NP-dur : justifie l'approche énumérative du générateur de l'exemple guidé 1 sur un domaine fini).

3. **Likhodedov A. & Sandholm T. (2003)** — *Auction mechanism for optimally trading off revenue and efficiency*. EC '03: Proceedings of the 4th ACM conference on Electronic commerce, San Diego, CA, USA, 212–213. DOI: [10.1145/779928.779964](https://doi.org/10.1145/779928.779964) — §3 (calibration revenue/efficiency : le vérificateur de l'exemple guidé 2 sépare la mesure de J du mécanisme candidat, ce qui reflète la décomposition optimisation/mesure introduite ici).

4. **Sandholm T. (2013)** — *Very-Large-Scale Generalized Combinatorial Multi-Attribute Auctions*. In: Vulkan N., Roth A. E., Neeman Z. (eds), *The Handbook of Market Design*, Oxford University Press, pp. 379–412. DOI: [10.1093/acprof:oso/9780199570515.003.0017](https://doi.org/10.1093/acprof:oso/9780199570515.003.0017) — §4 (sandbox large-scale AMD : le témoin d'impossibilité de l'exemple guidé 3 (paiement strictement positif + DSIC + IR + type=0) est l'ombre de la même structure qui, à grande échelle, justifie le recours à des solveurs dédiés — l'exemple pédagogique n'est pas anecdotique).

**Carnet suivant.** [GameTheory-16c — Extraction de revenu, DSIC/IR](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) -- la suite directe : l'organe juste de ce carnet (le vérificateur séparé) y est repris pour mesurer ce que coûte au designer l'ignorance de la dimension paiement.

**Carnets voisins.**

- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : principe de révélation, enchères de Vickrey, VCG et théorème d'impossibilité de Gibbard-Satterthwaite.
- [GameTheory-16d — Joueurs LLM](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) -- la même discipline (gel pré-enregistré, contrôles scriptés, vérificateur séparé) appliquée à des joueurs LLM réels.
- [GameTheory-03d — Deux espèces de flèches](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) -- la strate 7 que ce carnet ne franchit pas.
